<center> <img src = https://raw.githubusercontent.com/AndreyRysistov/DatasetsForPandas/main/hh%20label.jpg alt="drawing" style="width:400px;">

# <center> Проект: Анализ вакансий из HeadHunter
   

In [1]:
#импортирую нужные для работы библиотеки
import pandas as pd 
import psycopg2

In [2]:
#задаю переменные для подключения
DBNAME = 'project_sql'
USER = 'skillfactory'
PASSWORD = 'cCkxxLVrDE8EbvjueeMedPKt'
HOST = '84.201.134.129'
PORT = 5432

In [3]:
#создаю подключение к бд
connection = psycopg2.connect(
    dbname=DBNAME,
    user=USER,
    host=HOST,
    password=PASSWORD,
    port=PORT
)

# Юнит 3. Предварительный анализ данных

1. Напишите запрос, который посчитает количество вакансий в нашей базе (вакансии находятся в таблице vacancies). 

In [4]:
# считаю все вакансии по id
query_3_1 = f''' select count(id) from vacancies 
'''

In [5]:
# dataframe запроса и его вывод
df_3_1 = pd.read_sql_query(query_3_1, connection)
df_3_1

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3152312101.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3_1 = pd.read_sql_query(query_3_1, connection)


,count
0,49197


2. Напишите запрос, который посчитает количество работодателей (таблица employers). 

In [6]:
# считаю всех работодателей
query_3_2 = f''' select count(id) from employers
'''

In [7]:
#dataframe запроса и его вывод
df_3_2 = pd.read_sql_query(query_3_2, connection)
df_3_2

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2850338637.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3_2 = pd.read_sql_query(query_3_2, connection)


,count
0,23501


3. Посчитате с помощью запроса количество регионов (таблица areas).

In [8]:
# считаю все регионы
query_3_3 = f''' select count(id) from areas'''

In [9]:
# dataframe запроса и его вывод
df_3_3 = pd.read_sql_query(query_3_3, connection)
df_3_3

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3090575762.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3_3 = pd.read_sql_query(query_3_3, connection)


,count
0,1362


4. Посчитате с помощью запроса количество сфер деятельности в базе (таблица industries).

In [10]:
# считаю все сферы деятельности
query_3_4 = f''' select count(distinct industry_id) from employers_industries'''

In [11]:
# dataframe запроса и его вывод
df_3_4 = pd.read_sql_query(query_3_4, connection)
df_3_4

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3345285235.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_3_4 = pd.read_sql_query(query_3_4, connection)


,count
0,294


***

# Выводы по предварительному анализу данных
Всего у нас имеется 49197 вакансий, 23501 работодатель, 1362 региона и 294 сферы деятельности.

# Юнит 4. Детальный анализ вакансий

1. Напишите запрос, который позволит узнать, сколько (cnt) вакансий в каждом регионе (area).
Отсортируйте по количеству вакансий в порядке убывания.

In [12]:
# считаю кол-во вакансий в каждом регионе с помощью группировкт и join таблиц
query_4_1 = f'''select areas.name, count(vacancies.id) as cnt 
from vacancies 
join areas on vacancies.area_id = areas.id 
group by areas.id 
order by cnt desc '''

In [13]:
# dataframe запроса и его вывод
df_4_1 = pd.read_sql_query(query_4_1, connection)
df_4_1

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\632618379.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4_1 = pd.read_sql_query(query_4_1, connection)


,name,cnt
0,Москва,5333
1,Санкт-Петербург,2851
2,Минск,2112
3,Новосибирск,2006
4,Алматы,1892
...,...,...
764,Кизляр,1
765,Джизак,1
766,Эртиль,1
767,Арсеньев,1


2. Напишите запрос, чтобы определить у какого количества вакансий заполнено хотя бы одно из двух полей с зарплатой.

In [14]:
# считаю кол-во вакансий с хотя бы одним заполненным параметром используя where
query_4_2 = f'''select count(id) 
from vacancies 
where (not(salary_from is null) or not(salary_to is null)) '''

In [15]:
# dataframe запроса и его вывод
df_4_2 = pd.read_sql_query(query_4_2, connection)
df_4_2

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2142397265.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4_2 = pd.read_sql_query(query_4_2, connection)


,count
0,24073


3. Найдите средние значения для нижней и верхней границы зарплатной вилки. Округлите значения до целого.

In [16]:
# нахожу среднее значения зарплат используя avg для среднего и round для округления
query_4_3 = f'''select round(avg(salary_from)), 
round(avg(salary_to)) 
from vacancies '''

In [17]:
# dataframe запроса и его вывод
df_4_3 = pd.read_sql_query(query_4_3, connection)
df_4_3

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\757693061.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4_3 = pd.read_sql_query(query_4_3, connection)


,round,round
0,71065.0,110537.0


4. Напишите запрос, который выведет количество вакансий для каждого сочетания типа рабочего графика (schedule) и типа трудоустройства (employment), используемого в вакансиях. Результат отсортируйте по убыванию количества.


In [18]:
# считаю кол-во вакансий группируя сначала по одному, затем по другому параметру
query_4_4 = f'''select schedule, employment, count(id) 
from vacancies 
group by schedule, employment 
order by count(id) desc '''

In [19]:
#dataframe запроса и его вывод
df_4_4 = pd.read_sql_query(query_4_4, connection)
df_4_4

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\1819291790.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4_4 = pd.read_sql_query(query_4_4, connection)


,schedule,employment,count
0,Полный день,Полная занятость,35367
1,Удаленная работа,Полная занятость,7802
2,Гибкий график,Полная занятость,1593
3,Удаленная работа,Частичная занятость,1312
4,Сменный график,Полная занятость,940
5,Полный день,Стажировка,569
6,Вахтовый метод,Полная занятость,367
7,Полный день,Частичная занятость,347
8,Гибкий график,Частичная занятость,312
9,Полный день,Проектная работа,141


5. Напишите запрос, выводящий значения поля Требуемый опыт работы (experience) в порядке возрастания количества вакансий, в которых указан данный вариант опыта. 

In [20]:
# вывожу опыт и соответсвующее ему количество вакансий используя group by
query_4_5 = f'''select experience, count(id) 
from vacancies 
group by experience 
order by count(id) asc '''

In [21]:
# dataframe запроса и его вывод
df_4_5 = pd.read_sql_query(query_4_5, connection)
df_4_5

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2864564909.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_4_5 = pd.read_sql_query(query_4_5, connection)


,experience,count
0,Более 6 лет,1337
1,Нет опыта,7197
2,От 3 до 6 лет,14511
3,От 1 года до 3 лет,26152


***

# Выводы по детальному анализу вакансий
Больше всего вакансий в городе Москва, предположительно из-за наибольшего количества населения. Какие-то границы зарплаты указаны лишь у 24073 вакансий, при чем среднее значение нижней границы примерно 71065 рублей, верхней же границы примерно 110537.
По поводу комбинации рабочего графика и типа трудоустройства-на втором месте стоит удаленная работа и полная занятость, то есть тренд на удаленную работу спадать не собирается.
Большая часть ваканский требует опыт от 1 года до 3 лет, самие редко встречаемые-вакансии с требуемым опытом работы более 6 лет.

# Юнит 5. Анализ работодателей

1. Напишите запрос, который позволит узнать, какие работодатели находятся на первом и пятом месте по количеству вакансий.

In [22]:
# считаю вакансии по работодателям используя простой join для объеднения и group by
query_5_1 = f'''select employers.name, count(vacancies.id) from employers 
join vacancies
on employers.id = vacancies.employer_id 
group by employers.id 
order by count(vacancies.id) 
desc limit 5'''

In [23]:
# dataframe запроса и его вывод
df_5_1 = pd.read_sql_query(query_5_1, connection)
df_5_1

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\135811746.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5_1 = pd.read_sql_query(query_5_1, connection)


,name,count
0,Яндекс,1933
1,Ростелеком,491
2,Тинькофф,444
3,СБЕР,428
4,Газпром нефть,331


2. Напишите запрос, который для каждого региона выведет количество работодателей и вакансий в нём.
Среди регионов, в которых нет вакансий, найдите тот, в котором наибольшее количество работодателей.


In [24]:
#подсчитал количество используя left join чтобы записи с null справа также были
query_5_2 = f'''select areas.name, count(employers.id), count(vacancies.id) 
from areas  
left join vacancies on areas.id = vacancies.area_id 
left join employers on areas.id = employers.area  
group by areas.id 
having count(vacancies.id)=0 
order by count(employers.id) desc'''

In [25]:
# dataframe запроса и его вывод
df_5_2 = pd.read_sql_query(query_5_2, connection)
df_5_2

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3547941322.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5_2 = pd.read_sql_query(query_5_2, connection)


,name,count,count
0,Россия,410,0
1,Казахстан,207,0
2,Московская область,75,0
3,Краснодарский край,19,0
4,Беларусь,18,0
...,...,...,...
588,Лиман (Астраханская область),0,0
589,Хоринск,0,0
590,Старица,0,0
591,Ессентукская,0,0


3. Для каждого работодателя посчитайте количество регионов, в которых он публикует свои вакансии. Отсортируйте результат по убыванию количества.


In [26]:
# подсчитал кол-во регионов используя distinct для подсчета именно регионов и таблицу vacancies для отбора регионов
query_5_3 = f'''select e1.name, count(distinct(v.area_id)) 
from employers as e1 
join vacancies as v on e1.id = v.employer_id  
group by e1.id 
order by count(distinct(v.area_id)) desc '''

In [27]:
# dataframe запроса и его вывод
df_5_3 = pd.read_sql_query(query_5_3, connection)
df_5_3

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\681425330.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5_3 = pd.read_sql_query(query_5_3, connection)


,name,count
0,Яндекс,181
1,Ростелеком,152
2,Спецремонт,116
3,Поляков Денис Иванович,88
4,ООО ЕФИН,71
...,...,...
14901,НПП Авиатрон,1
14902,Центр дистанционных торгов,1
14903,Городские Телекоммуникационные Системы,1
14904,"Введенский, Отель",1


4. Напишите запрос для подсчёта количества работодателей, у которых не указана сфера деятельности. 

In [28]:
#подсчет кол-ва с null, использовал left join чтобы не отсеивать записи с null справа 
query_5_4 = f'''select count(e1.id) 
from employers as e1 
left join employers_industries as e_i on e1.id = e_i.employer_id  
where e_i.employer_id is null '''

In [29]:
# dataframe запроса и его вывод
df_5_4 = pd.read_sql_query(query_5_4, connection)
df_5_4

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3445989836.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5_4 = pd.read_sql_query(query_5_4, connection)


,count
0,8419


5. Напишите запрос, чтобы узнать название компании, находящейся на третьем месте в алфавитном списке (по названию) компаний, у которых указано четыре сферы деятельности. 

In [30]:
# вывел компания, используя простой join и having для условия с агр. функцией 
query_5_5 = f'''select e.name 
from employers as e 
join employers_industries as e_i on e.id = e_i.employer_id 
group by e.id  
having count(e_i.employer_id)=4 
order by e.name '''

In [31]:
# dataframe запроса и его вывод
df_5_5 = pd.read_sql_query(query_5_5, connection)
df_5_5

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\131240259.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5_5 = pd.read_sql_query(query_5_5, connection)


,name
0,101 Интернет
1,21vek.by
2,2ГИС
3,2К
4,4 пикселя +
...,...
1133,ЮРИОН
1134,ЮТИП Технологии
1135,ЯКласс
1136,ЯрНео


6. С помощью запроса выясните, у какого количества работодателей в качестве сферы деятельности указана Разработка программного обеспечения.


In [32]:
# подсчитал кол-во используя like для условия со строкой
query_5_6 = f'''select count(employer_id)  
from employers_industries 
join industries on industry_id = id 
where name like 'Разработка программного обеспечения' '''

In [33]:
# dataframe запроса и его вывод
df_5_6 = pd.read_sql_query(query_5_6, connection)
df_5_6

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\50951690.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_5_6 = pd.read_sql_query(query_5_6, connection)


,count
0,3553


7. Для компании «Яндекс» выведите список регионов-миллионников, в которых представлены вакансии компании, вместе с количеством вакансий в этих регионах. Также добавьте строку Total с общим количеством вакансий компании. Результат отсортируйте по возрастанию количества.

Список городов-милионников надо взять [отсюда](https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8). 

Если возникнут трудности с этим задание посмотрите материалы модуля  PYTHON-17. Как получать данные из веб-источников и API. 

In [34]:
#импорт библиотек
from bs4 import BeautifulSoup
import requests
#задал url который будем парсить
url = 'https://ru.wikipedia.org/wiki/%D0%93%D0%BE%D1%80%D0%BE%D0%B4%D0%B0-%D0%BC%D0%B8%D0%BB%D0%BB%D0%B8%D0%BE%D0%BD%D0%B5%D1%80%D1%8B_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B8'
#создал переменнуб для получения ответа
response = requests.get(url)
#пропарсил страницу, выделил таблицу и нашел все ссылки для списка городов
page = BeautifulSoup(response.text, 'html.parser').table.find_all('a')
list_mil_city = []
#прошелся по составленному списку и записал все города используя тег "text" 
for i in page:
    if i.text[0]!='[' and i.text[0]!='п':
        list_mil_city.append(i.text)


In [35]:
#написал запрос для вывода регионов и кол-ва вакансий в них, чтобы отсеить ненужные добавил еще одну таблицу в соединении и  условие
query_5_7 = f'''(select a.name, count(v.id) 
from areas a 
join vacancies v  on a.id = v.area_id 
join employers e on v.employer_id = e.id 
where e.name like 'Яндекс' group by a.id )
order by 2'''

In [36]:
# dataframe запроса
predf_5_7 = pd.read_sql_query(query_5_7, connection)
#создал новый df и отфильтровал нужные города
df_5_7 = predf_5_7[predf_5_7['name'].isin(list_mil_city)]
#добавил строку total в конец df
df_5_7.loc[len(df_5_7.index)]= ('Total', sum(df_5_7['count']))
#вывод
df_5_7


C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\993894728.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  predf_5_7 = pd.read_sql_query(query_5_7, connection)
C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\993894728.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_5_7.loc[len(df_5_7.index)]= ('Total', sum(df_5_7['count']))


,name,count
155,Омск,21
159,Челябинск,22
163,Красноярск,23
165,Волгоград,24
168,Ростов-на-Дону,25
169,Казань,25
170,Пермь,25
171,Самара,26
172,Уфа,26
174,Краснодар,30


***

# Выводы по анализу работодателей
Больше всего вакансий предоставляет компания Яндекс. Ее вакансии представлены во всех 16 городах-миллионниках и в 181 регионе. Регионом же с наибольшим количеством работодателей и с нулевым количеством вакансий яляется Россия, так как все компании ищут сотрудников только по конкретным регионам. Всего 8419 работодателей не указывают свою сферу детятельности. 3553 компании, представленные в таблице, указали в качестве сферы деятельности «Разработка программного обеспечения».

# Юнит 6. Предметный анализ

1. Сколько вакансий имеет отношение к данным?

Считаем, что вакансия имеет отношение к данным, если в её названии содержатся слова 'data' или 'данн'.

*Подсказка: Обратите внимание, что названия вакансий могут быть написаны в любом регистре.* 


In [37]:
#подсчитал кол-во используя ilike для фильтрации строк без строгости по регистру
query_6_1 = f'''select count(id) 
from vacancies 
where name ilike '%data%' or name ilike '%данн%'  '''

In [38]:
# dataframe запроса и его вывод
df_6_1 = pd.read_sql_query(query_6_1, connection)
df_6_1

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\1395909966.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6_1 = pd.read_sql_query(query_6_1, connection)


,count
0,1771


2. Сколько есть подходящих вакансий для начинающего дата-сайентиста? 
Будем считать вакансиями для дата-сайентистов такие, в названии которых есть хотя бы одно из следующих сочетаний:
* 'data scientist'
* 'data science'
* 'исследователь данных'
* 'ML' (здесь не нужно брать вакансии по HTML)
* 'machine learning'
* 'машинн%обучен%'

** В следующих заданиях мы продолжим работать с вакансиями по этому условию.*

Считаем вакансиями для специалистов уровня Junior следующие:
* в названии есть слово 'junior' *или*
* требуемый опыт — Нет опыта *или*
* тип трудоустройства — Стажировка.
 

In [39]:
#подсчитал кол-о используя сложное условие для фильтрации строк
query_6_2 = f'''select count(id) 
from vacancies 
where (name ilike '%data scientist%' 
or name ilike '%data science%' 
or name ilike '%исследователь данных%' 
or name ilike '%ML%'  
or name ilike '%machine learning%' 
or name ilike '%машинн%обучен%') 
and (name ilike '%junior%' 
or experience ilike 'Нет опыта' 
or employment ilike '%Стажировка%') 
and not(name ilike '%HTML%')  '''

In [40]:
# dataframe запроса и его вывод
df_6_2 = pd.read_sql_query(query_6_2, connection)
df_6_2

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3166471988.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6_2 = pd.read_sql_query(query_6_2, connection)


,count
0,51


3. Сколько есть вакансий для DS, в которых в качестве ключевого навыка указан SQL или postgres?

** Критерии для отнесения вакансии к DS указаны в предыдущем задании.*

In [41]:

#опять кол-во и опять длинное условие для фильтрации строк
query_6_3 = f'''select count(id) from vacancies 
where (name ilike '%data scientist%' 
or name ilike '%data science%' 
or name ilike '%исследователь данных%' 
or name ilike '%ML%'  
or name ilike '%machine learning%' 
or name ilike '%машинн%обучен%') 
and not(name ilike '%HTML%')  
and (key_skills ilike '%SQL%' 
or key_skills ilike '%postgres%') '''

In [42]:
#dataframe запроса и его вывод
df_6_3 = pd.read_sql_query(query_6_3, connection)
df_6_3

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\1791643786.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6_3 = pd.read_sql_query(query_6_3, connection)


,count
0,229


4. Проверьте, насколько популярен Python в требованиях работодателей к DS.Для этого вычислите количество вакансий, в которых в качестве ключевого навыка указан Python.

** Это можно сделать помощью запроса, аналогичного предыдущему.*

In [43]:

#опять кол-во и опять длинное условие для фильтрации строк
query_6_4 = f'''select count(id) 
from vacancies
where (name ilike '%data scientist%' 
or name ilike '%data science%' 
or name ilike '%исследователь данных%' 
or name ilike '%ML%'  
or name ilike '%machine learning%' 
or name ilike '%машинн%обучен%') 
and not(name ilike '%HTML%')  
and (key_skills ilike '%python%' ) '''

In [44]:
# dataframe запроса и его вывод
df_6_4 = pd.read_sql_query(query_6_4, connection)
df_6_4

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\38975871.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6_4 = pd.read_sql_query(query_6_4, connection)


,count
0,357


5. Сколько ключевых навыков в среднем указывают в вакансиях для DS?
Ответ округлите до двух знаков после точки-разделителя.

In [45]:
# использую формулу для подсчета различных навыков в одной строке с помощью длины строки и длины без разд. символа
query_6_5 = f''' select round(avg(length(key_skills)-length(replace(key_skills, CHR(9), ''))+1), 2) 
from vacancies where not(key_skills is NULL) 
and (name ilike '%data scientist%' or name ilike '%data science%'
or name ilike '%исследователь данных%' or name like '%ML%'  
or name ilike '%machine learning%' or name ilike '%машинн%обучен%') and not(name ilike '%HTML%')'''

In [46]:
# dataframe запроса и его вывод
df_6_5 = pd.read_sql_query(query_6_5, connection)
df_6_5

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3107599176.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6_5 = pd.read_sql_query(query_6_5, connection)


,round
0,6.41


6. Напишите запрос, позволяющий вычислить, какую зарплату для DS в **среднем** указывают для каждого типа требуемого опыта (уникальное значение из поля *experience*). 

При решении задачи примите во внимание следующее:
1. Рассматриваем только вакансии, у которых заполнено хотя бы одно из двух полей с зарплатой.
2. Если заполнены оба поля с зарплатой, то считаем зарплату по каждой вакансии как сумму двух полей, делённую на 2. Если заполнено только одно из полей, то его и считаем зарплатой по вакансии.
3. Если в расчётах участвует null, в результате он тоже даст null (посмотрите, что возвращает запрос select 1 + null). Чтобы избежать этой ситуацию, мы воспользуемся функцией [coalesce](https://postgrespro.ru/docs/postgresql/9.5/functions-conditional#functions-coalesce-nvl-ifnull), которая заменит null на значение, которое мы передадим. Например, посмотрите, что возвращает запрос `select 1 + coalesce(null, 0)`

Выясните, на какую зарплату в среднем может рассчитывать дата-сайентист с опытом работы от 3 до 6 лет. Результат округлите до целого числа. 

In [47]:
#использую coalesce для условия о подсчете зп и обработки случаев с null в одном из полей
query_6_6 = f''' select distinct(experience),  
round( avg( coalesce( (salary_to+salary_from)/2, salary_to, salary_from) ), 0 )  
from vacancies where (not(salary_from is NULL) or not(salary_to is NULL)) 
and (name ilike '%data scientist%' or name ilike '%data science%'
or name ilike '%исследователь данных%' or name ilike '%машинн%обучен%'  
or name ilike '%machine learning%' or (name like '%ML%' and not (name like '%HTML%')))
group by 1'''

In [48]:
# dataframe запроса и его вывод
df_6_6 = pd.read_sql_query(query_6_6, connection)
df_6_6

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2288196939.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_6_6 = pd.read_sql_query(query_6_6, connection)


,experience,round
0,Нет опыта,74643.0
1,От 1 года до 3 лет,139675.0
2,От 3 до 6 лет,243115.0


***

# Выводы по предметному анализу
Всего связанных с данными вакансий 1771. Если говорить о вакансиях для начинающего data-science специалиста, то их не так много: всего 51 вакансия. Скорее всего это лишь пока, так как тренд данные растет с учетом развития нейросетей. 229 вакансий требуют знания sql, 357 требуют знания python. В среднем просят 6-7 навыков для данных должностей. Для людей без опыта предлагается зп в среднем 74643 рубля, для людей с опытом от 1 года и до 3 лет-139675 рублей и для людей с опытом 3-6 лет - 243115 рублей. 

# Общий вывод по проекту

In [49]:
#запрос для отыскания кол-ва вакансий с типом "Стажировка", без зарплаты и его вывод
query_dop_1 = f''' select count(id) 
from vacancies 
where employment like 'Стажировка' 
and salary_from is NULL 
and salary_to is NULL '''
df_dop_1 = pd.read_sql_query(query_dop_1, connection)
df_dop_1

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2749656210.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_1 = pd.read_sql_query(query_dop_1, connection)


,count
0,407


In [50]:
#кол-во вакансий от каждого работодателя
query_dop_2 = f''' select e.name,  count(v.id) 
from employers e 
join vacancies v on v.employer_id = e.id 
where v.salary_from is NULL and v.salary_to is NULL group by 1 order by 2 desc'''
df_dop_2 = pd.read_sql_query(query_dop_2, connection)
df_dop_2

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\1116118854.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_2 = pd.read_sql_query(query_dop_2, connection)


,name,count
0,Яндекс,438
1,Тинькофф,437
2,СБЕР,397
3,Газпром нефть,329
4,ИК СИБИНТЕК,259
...,...,...
5979,ВК Кузбасская ярмарка,1
5980,Placestart,1
5981,Techno Diasoft,1
5982,Webjox,1


In [51]:
#срежняя зп по регионам
query_dop_3 = f''' select a.name,  round(avg(coalesce((v.salary_to+v.salary_from)/2, v.salary_to, v.salary_from)), 0)  

from areas a 
join vacancies v 
on v.area_id = a.id 
where (not(salary_from is NULL) or not(salary_to is NULL)) 
group by 1 
order by 2 desc'''
df_dop_3 = pd.read_sql_query(query_dop_3, connection)
df_dop_3

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2748753978.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_3 = pd.read_sql_query(query_dop_3, connection)


,name,round
0,Германия,452751.0
1,Болгария,324348.0
2,Испания,300000.0
3,Черногория,288589.0
4,Литва,263019.0
...,...,...
666,Урень,13800.0
667,Ош,12743.0
668,Кобрин,12389.0
669,Степногорск,12205.0


In [52]:
#кол-во работодателей по регионам
query_dop_4 = f''' select a.name,  count(e.id) from areas a 
join employers e on e.area = a.id 
group by 1 
order by 2 desc'''
df_dop_4 = pd.read_sql_query(query_dop_4, connection)
df_dop_4

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3462395613.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_4 = pd.read_sql_query(query_dop_4, connection)


,name,count
0,Москва,5864
1,Санкт-Петербург,2217
2,Минск,1115
3,Алматы,721
4,Екатеринбург,609
...,...,...
649,Узловая,1
650,Междуреченск,1
651,Березовский (Свердловская область),1
652,Лебедянь,1


In [53]:
#работодатели, их регион и их средняя зп
query_dop_5 = f''' select e.name, a.name,  round(avg(coalesce((v.salary_to+v.salary_from)/2, v.salary_to, v.salary_from)), 0)  
from employers e 
join vacancies v on v.employer_id = e.id
join areas a on a.id = e.area 
where (not(salary_from is NULL) or not(salary_to is NULL)) 
group by 1,2 
order by 3 desc'''
df_dop_5 = pd.read_sql_query(query_dop_5, connection)
df_dop_5

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\2019217885.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_5 = pd.read_sql_query(query_dop_5, connection)


,name,name,round
0,Alpha Personnel,Санкт-Петербург,1000000.0
1,ЦТП,Москва,650000.0
2,RuWork,Москва,555261.0
3,Петухова Алия,Казань,552700.0
4,Пикабу,Москва,550000.0
...,...,...,...
10571,Авсянников Василий,Брест,900.0
10572,Эксперт,Хабаровск,800.0
10573,Вьюэво,Санкт-Петербург,300.0
10574,DauInvest,Нур-Султан,65.0


In [54]:
#сферы яндекса
query_dop_6 = f''' select e.name,  count(e_i.industry_id) from employers e 
join employers_industries e_i on e.id = e_i.employer_id
where e.name like 'Яндекс'
group by e.id'''
df_dop_6 = pd.read_sql_query(query_dop_6, connection)
df_dop_6

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\613315435.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_6 = pd.read_sql_query(query_dop_6, connection)


,name,count
0,Яндекс,1


In [55]:

#кол-во вакансий по data-science
query_dop_7 = f'''select count(id) from vacancies 
where (name ilike '%data scientist%' or name ilike '%data science%' 
or name ilike '%исследователь данных%' 
or name ilike '%ML%'  
or name ilike '%machine learning%' 
or name ilike '%машинн%обучен%') 
and not(name ilike '%HTML%')  '''
df_dop_7 = pd.read_sql_query(query_dop_7, connection)
df_dop_7

C:\Users\dsaut\AppData\Local\Temp\ipykernel_544\3452358169.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dop_7 = pd.read_sql_query(query_dop_7, connection)


,count
0,536


# Вывод по анализу
В анализе учавствовали 49197 вакансий от 23501 работодателя и 1362 регионов.
Больше всего вакансий представлено в крупных городах с развитой технологической и экономической сферой. Менее чем в половине вакансий указаны хоть какие-то данные о зарплате. Только это не неоплачиваемые стажировки (их всего 401), а, похоже, либо намеренное умалчивание зарплаты ради большего отклика, либо компания пока не знает какую цену устанавливать из-за маленького опыта в работе с данными сотрудниками. Наибольшее таких сотрудников у крупных компаний, таких как Яндекс и Тинькофф.
Касательно формата работы, большая часть компаний ищут сотрудников на полную занятость и полный рабочий день. Конечно на втором месте идет удаленная работа, но это говорит о том, что времена ковида прошли и компании осознали преимущества работы очно. 
Касательно опыта, больше всего ищут работников с опытом работы от 1 года до трех лет, так как данные люди это скорее всего молодые люди, только закончившие универ, но уже при этом имеющие хоть какой-то опыт работы и понимание своего рода деятельности. Меньше всего ищут работников с опытом работы более 6 лет, лишний раз подчеркивая нужду рынка в молодых специалистах.
Среди всех регионов наибольшей средней зп обладает Германия, это связано с дорогой стоимостью жизни. Но вот среди компаний лидирует Петербуржская компания "Alpha Personnel" cо зредней зп 1000000. 
Больше всего работодателей представлено в Москве, так как там наибольшее количество населения. Больше всего вакансий предоставляет Яндекс, несмотря на всего лишь одну сферу деятельности, хоть и весьма обширную. Около трети работодателей не указывают свою сферу деятельности. 3553 работодателя указали в сфере деятельности "Разработка ПО", что составляет 15% от общего кол-ва работодателей, что говорит о наполненности рынка IT компаниями и еще раз подчеркивает перспективность IT направления в нашей стране.
По аналитике данных вакансий всего 536, что является не малым количеством, но и не большим, но прогнозирую, что с приходом нейросетей данных вакансий станет намного больше.